# Solving Pickle Doomsday

In [1]:
from oggm import cfg, utils
from oggm import workflow
from oggm import DEFAULT_BASE_URL
import oggmzarr.datacube.convert as convert
import xarray as xr

In [2]:
cfg.initialize(logging_level="ERROR")

cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["continue_on_error"] = True
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='oggm-geozarr', reset=True)
rgi_ids = ['RGI60-11.00897', "RGI60-06.00377"] # HEF, Bruarjoekull 

DEFAULT_BASE_URL

2026-04-30 23:55:07: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-04-30 23:55:07: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-04-30 23:55:07: oggm.cfg: Multiprocessing: using all available processors (N=20)
2026-04-30 23:55:07: oggm.cfg: Multiprocessing switched ON after user settings.
2026-04-30 23:55:07: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.


'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup/'

In [3]:
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    prepro_base_url=DEFAULT_BASE_URL,  # where to fetch the data?
    from_prepro_level=4,  # what kind of data? 
    prepro_border=80  # how big of a map?
)
gdir = gdirs[0]

2026-04-30 23:55:08: oggm.workflow: init_glacier_directories from prepro level 4 on 2 glaciers.
2026-04-30 23:55:08: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 2 glaciers


In [4]:
pickle_paths = convert.get_pickle_paths(gdir)
pickle_paths

[PosixPath('model_flowlines_dyn_melt_f_calib.pkl'),
 PosixPath('model_flowlines.pkl'),
 PosixPath('inversion_output.pkl'),
 PosixPath('inversion_input.pkl'),
 PosixPath('inversion_flowlines.pkl'),
 PosixPath('downstream_line.pkl')]

In [5]:
def text_to_dict(lines):
    v_text = ["| variable| type | description |\n|---|---|---|"]
    v_text.append(f"{"\n  - ".join([f"<b>{i}</b>: {str(j)[7:-1]}" for i,j in lines.items()])}")
    return v_text

In [6]:
pickle_data = convert.get_pickle_data(pickle_paths, gdir, type_only=False)
pickle_types = convert.get_pickle_data(pickle_paths, gdir, type_only=True)


model_flowlines_dyn_melt_f_calib not in cfg.BASENAMES.
Pickle model_flowlines_dyn_melt_f_calib of type <class 'str'> not parseable.
'Centerline' object has no attribute 'items'
Pickle downstream_line of type <class 'str'> not parseable.
model_flowlines_dyn_melt_f_calib not in cfg.BASENAMES.
Pickle model_flowlines_dyn_melt_f_calib of type <class 'str'> not parseable.
'Centerline' object has no attribute 'items'
Pickle downstream_line of type <class 'str'> not parseable.


Generate GH markdown for pickle types.

In [7]:
# pprint.pp(pickle_data, sort_dicts=True)
text = []
for k,v in pickle_types.items():
    if isinstance(v, list):
        if isinstance(v[0], dict):
            v_text = text_to_dict(v[0])
        else:
            v_text = f"{v}"
    elif isinstance(v, dict):
        v_text = text_to_dict(v)
    else:
        v_text = f"{v}"
    text.append(f"| <b>{k}</b> | {v_text}| |\n")
print("\n".join(text))

| <b>model_flowlines</b> | [<class 'oggm.core.flowline.MixedBedFlowline'>]| |

| <b>inversion_output</b> | ['| variable| type | description |\n|---|---|---|', "<b>dx</b>: 'float'\n  - <b>flux_a0</b>: 'numpy.ndarray'\n  - <b>width</b>: 'numpy.ndarray'\n  - <b>slope_angle</b>: 'numpy.ndarray'\n  - <b>is_rectangular</b>: 'numpy.ndarray'\n  - <b>is_trapezoid</b>: 'numpy.ndarray'\n  - <b>flux</b>: 'numpy.ndarray'\n  - <b>is_last</b>: 'bool'\n  - <b>hgt</b>: 'numpy.ndarray'\n  - <b>invert_with_trapezoid</b>: 'bool'\n  - <b>thick</b>: 'numpy.ndarray'\n  - <b>volume</b>: 'numpy.ndarray'"]| |

| <b>inversion_input</b> | ['| variable| type | description |\n|---|---|---|', "<b>dx</b>: 'float'\n  - <b>flux_a0</b>: 'numpy.ndarray'\n  - <b>width</b>: 'numpy.ndarray'\n  - <b>slope_angle</b>: 'numpy.ndarray'\n  - <b>is_rectangular</b>: 'numpy.ndarray'\n  - <b>is_trapezoid</b>: 'numpy.ndarray'\n  - <b>flux</b>: 'numpy.ndarray'\n  - <b>is_last</b>: 'bool'\n  - <b>hgt</b>: 'numpy.ndarray'\n  - <b>invert_

Convert supported pickles to data tree (currently only `inversion_output` and `inversion_input`)

In [8]:
data_tree = convert.convert_pickle_to_datatree(pickle_data=pickle_data)
data_tree

<xarray.DataTree>
Group: /
├── Group: /inversion_output
│       Dimensions:                (flux_a0: 50, width: 50, slope_angle: 50,
│                                   is_rectangular: 50, is_trapezoid: 50, flux: 50,
│                                   hgt: 50, thick: 50, volume: 50)
│       Coordinates:
│         * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
│         * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
│         * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
│         * is_rectangular         (is_rectangular) bool 50B False False ... False False
│         * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
│         * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
│         * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
│         * thick                  (thick) float64 400B 33.17 35.88 ... 38.42 27.32
│         * volume                 (volume) float64 400B 2.5e+06 2.068e+06 ... 1.257e+06
│       Data variables:
│           dx                     float64 8B 100.0
│           is_last                bool 1B True
│           invert_with_trapezoid  bool 1B True
└── Group: /inversion_input
        Dimensions:                (flux_a0: 50, width: 50, slope_angle: 50,
                                    is_rectangular: 50, is_trapezoid: 50, flux: 50,
                                    hgt: 50)
        Coordinates:
          * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
          * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
          * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
          * is_rectangular         (is_rectangular) bool 50B False False ... False False
          * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
          * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
          * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
        Data variables:
            dx                     float64 8B 100.0
            is_last                bool 1B True
            invert_with_trapezoid  bool 1B True

Let's now tackle more complex pickles.

In [31]:
pickle = gdir.read_pickle("downstream_line")
pickle

{'full_line': None,
 'downstream_line': <LINESTRING (196 80, 197.414 78.586, 199.236 77.764, 201.057 76.943, 202.471...>,
 'bedshapes': array([0.00199096, 0.00201091, 0.00196549, 0.00193507, 0.00191356,
        0.0018767 , 0.00181451, 0.00178074, 0.0016981 , 0.00162066,
        0.00149715, 0.00140146, 0.00136911, 0.0013935 , 0.00144497,
        0.00156016, 0.00167799, 0.00176282, 0.00182802, 0.00186648,
        0.001871  , 0.00182576, 0.00177926, 0.00173999, 0.00173839,
        0.00175625, 0.00179726, 0.00182617, 0.00181438, 0.00177377,
        0.00170936, 0.00168871, 0.00169273, 0.00173031, 0.00181565,
        0.00190387, 0.00190635, 0.00189614, 0.00187313, 0.00180662,
        0.00175849, 0.00175849, 0.00175849, 0.00175849, 0.00175849,
        0.00175849, 0.00175849, 0.00175849, 0.00175849, 0.00175849]),
 'surface_h': array([2430.45166278, 2427.46685638, 2423.07340085, 2418.50981606,
        2413.8558899 , 2408.86104904, 2403.89253962, 2399.1678485 ,
        2394.85408923, 2391.213967

This pickle contains a shapely LINESTRING object

In [26]:
downstream_line = pickle["downstream_line"]
type(downstream_line)

shapely.geometry.linestring.LineString

In [27]:
from shapely.geometry import mapping
mapping(downstream_line)
# import geopandas as gpd
# from shapely.geometry import LineString

# # Create a GeoDataFrame
# gdf = gpd.GeoDataFrame(geometry=[LineString([(0, 0), (1, 1)])])

# # Export to file
# gdf.to_file("line.geojson", driver="GeoJSON")


{'type': 'LineString',
 'coordinates': ((196.0, 80.0),
  (197.4142135623731, 78.58578643762691),
  (199.23551672757765, 77.76448327242237),
  (201.0568198927822, 76.94318010721783),
  (202.47103345515526, 75.52896654484472),
  (204.2923366203598, 74.7076633796402),
  (205.7065501827329, 73.2934498172671),
  (207.52785334793745, 72.47214665206256),
  (209.46889249255338, 72.0),
  (211.198446748901, 71.0),
  (213.198446748901, 71.0),
  (215.198446748901, 71.0),
  (217.198446748901, 71.0),
  (219.198446748901, 71.0),
  (221.198446748901, 71.0),
  (223.198446748901, 71.0),
  (225.198446748901, 71.0),
  (227.198446748901, 71.0),
  (229.198446748901, 71.0),
  (231.198446748901, 71.0),
  (233.198446748901, 71.0),
  (235.198446748901, 71.0),
  (237.198446748901, 71.0),
  (239.198446748901, 71.0),
  (241.198446748901, 71.0),
  (243.18915419898423, 70.81084580101576),
  (245.01045736418877, 69.98954263581123),
  (246.74627952341825, 69.0),
  (248.47583377976588, 68.0),
  (250.1270061851359, 66.8

In [104]:
import shapely
import numpy as np
pickle = gdir.read_pickle("downstream_line")

coordinates = xr.DataArray(np.array(mapping(downstream_line)["coordinates"]))
coordinates
# linestring = shapely.LineString(coordinates)
# assert linestring.equals(pickle["downstream_line"])
pickle["downstream_line"] = coordinates
# pickle
pickle_data["downstream_line"] = pickle
# data_tree = xr.DataTree()
data_tree = convert.add_datacube(data_tree=data_tree, datacubes=pickle, datacube_name="downstream_line", overwrite=True)
# data_tree = convert.convert_pickle_to_datatree(pickle_data=pickle_data)
data_tree.downstream_line.downstream_line

<xarray.DataArray 'downstream_line' (dim_0: 50, dim_1: 2)> Size: 800B
array([[196.        ,  80.        ],
       [197.41421356,  78.58578644],
       [199.23551673,  77.76448327],
       [201.05681989,  76.94318011],
       [202.47103346,  75.52896654],
       [204.29233662,  74.70766338],
       [205.70655018,  73.29344982],
       [207.52785335,  72.47214665],
       [209.46889249,  72.        ],
       [211.19844675,  71.        ],
       [213.19844675,  71.        ],
       [215.19844675,  71.        ],
       [217.19844675,  71.        ],
       [219.19844675,  71.        ],
       [221.19844675,  71.        ],
       [223.19844675,  71.        ],
       [225.19844675,  71.        ],
       [227.19844675,  71.        ],
       [229.19844675,  71.        ],
       [231.19844675,  71.        ],
...
       [251.54121975,  65.45878025],
       [252.95543331,  64.04456669],
       [254.36964687,  62.63035313],
       [255.78386043,  61.21613957],
       [257.198074  ,  59.801926  ],
       [258.61228756,  58.38771244],
       [260.02650112,  56.97349888],
       [261.44071468,  55.55928532],
       [262.85492825,  54.14507175],
       [264.26914181,  52.73085819],
       [265.68335537,  51.31664463],
       [267.09756893,  49.90243107],
       [268.5117825 ,  48.4882175 ],
       [269.33308566,  46.66691434],
       [270.15438883,  44.84561117],
       [271.        ,  43.03580484],
       [272.        ,  41.30625058],
       [273.25212637,  39.74787363],
       [274.66633993,  38.33366007],
       [276.08055349,  36.91944651]])
Dimensions without coordinates: dim_0, dim_1

# Export and validate

In [103]:
convert.write_zarr(data_tree=data_tree, storage_directory=f"{gdir.rgi_id}.zarr",)

Check we can read the generated zarr

In [105]:
stream_cube = xr.open_datatree(
    f"{gdir.rgi_id}.zarr",
    group=None,
    chunks={},
    engine="zarr",
    consolidated=True,
    decode_cf=True,
)
stream_cube

<xarray.DataTree>
Group: /
├── Group: /downstream_line
│   │   Dimensions:          (dim_0: 50, dim_1: 2, bedshapes: 50, surface_h: 50, w0s: 50)
│   │   Coordinates:
│   │     * bedshapes        (bedshapes) float64 400B 0.001991 0.002011 ... 0.001758
│   │     * surface_h        (surface_h) float64 400B 2.43e+03 2.427e+03 ... 2.153e+03
│   │     * w0s              (w0s) float64 400B 179.0 165.9 176.1 ... 208.9 208.9 208.9
│   │   Dimensions without coordinates: dim_0, dim_1
│   │   Data variables:
│   │       downstream_line  (dim_0, dim_1) float64 800B dask.array<chunksize=(50, 2), meta=np.ndarray>
│   └── Group: /downstream_line/full_line
├── Group: /inversion_input
│       Dimensions:                (flux: 50, flux_a0: 50, hgt: 50, is_rectangular: 50,
│                                   is_trapezoid: 50, slope_angle: 50, width: 50)
│       Coordinates:
│         * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
│         * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
│         * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
│         * is_rectangular         (is_rectangular) bool 50B False False ... False False
│         * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
│         * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
│         * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
│       Data variables:
│           dx                     float64 8B ...
│           invert_with_trapezoid  bool 1B ...
│           is_last                bool 1B ...
└── Group: /inversion_output
        Dimensions:                (flux: 50, flux_a0: 50, hgt: 50, is_rectangular: 50,
                                    is_trapezoid: 50, slope_angle: 50, thick: 50,
                                    volume: 50, width: 50)
        Coordinates:
          * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
          * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
          * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
          * is_rectangular         (is_rectangular) bool 50B False False ... False False
          * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
          * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
          * thick                  (thick) float64 400B 33.17 35.88 ... 38.42 27.32
          * volume                 (volume) float64 400B 2.5e+06 2.068e+06 ... 1.257e+06
          * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
        Data variables:
            dx                     float64 8B ...
            invert_with_trapezoid  bool 1B ...
            is_last                bool 1B ...

In [11]:
stream_cube.inversion_input

<xarray.DataTree 'inversion_input'>
Group: /inversion_input
    Dimensions:                (flux: 50, flux_a0: 50, hgt: 50, is_rectangular: 50,
                                is_trapezoid: 50, slope_angle: 50, width: 50)
    Coordinates:
      * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
      * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
      * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
      * is_rectangular         (is_rectangular) bool 50B False False ... False False
      * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
      * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
      * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
    Data variables:
        dx                     float64 8B ...
        invert_with_trapezoid  bool 1B ...
        is_last                bool 1B ...